# 01 · Análise Exploratória dos Dados

Exploração da base longitudinal de usuários: distribuição das classes, perfil de
atividade, vocabulário por grupo e padrões temporais.

> **Escopo deste notebook:** apenas exploração e relatório. Toda lógica de
> produção vive em `src/` — o notebook consome o pipeline, não o reimplementa.

> ⚠️ **Nunca exiba texto bruto de tweets neste notebook.** Os outputs são
> removidos pelo `nbstripout` antes do commit, mas a disciplina de não
> imprimir conteúdo identificável vem primeiro.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

from config.paths import get_paths
from visualization.theme import apply_theme

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
apply_theme()
plt.rcParams["figure.dpi"] = 120

PATHS = get_paths()

## 1. Carregamento

In [ ]:
from data.reader import read_parquet

tweets = read_parquet(PATHS.data.tweets_labeled)
labels = read_parquet(PATHS.data.user_labels)
features = read_parquet(PATHS.data.user_features)

print(f"Tweets:   {tweets.height:>8,}")
print(f"Usuários: {labels.height:>8,}")
print(f"Atributos:{features.width:>8,}")

## 2. Distribuição das classes

O desbalanceamento é o contexto que determina se um F1 de 0,75 é bom ou apenas
reflete a classe majoritária.

In [ ]:
from visualization.distributions import plot_class_distribution

plot_class_distribution(labels)
plt.show()

labels.group_by("user_label").len().sort("len", descending=True)

## 3. Perfil de atividade

Volume de publicação, janela de observação e dias ativos. Usuários com histórico
curto não sustentam as features de tendência — e a etapa `features` os filtra.

In [ ]:
from visualization.distributions import plot_user_activity

plot_user_activity(features)
plt.show()

features.select(["n_tweets", "span_days", "active_days"]).describe()

## 4. Vocabulário por classe

Um painel por classe: o que interessa é o **contraste** de vocabulário entre os
grupos, que barras empilhadas esconderiam.

In [ ]:
from visualization.distributions import plot_ngrams, plot_word_frequency

plot_word_frequency(tweets, labels, top_n=20)
plt.show()

plot_ngrams(tweets, labels, n=2, top_n=15)
plt.show()

## 5. Padrões temporais

Nenhuma destas figuras pode ser produzida a partir de tweets isolados — elas são
a justificativa visual da abordagem centrada no usuário.

In [ ]:
from visualization.temporal_plots import (
    plot_activity_heatmap,
    plot_circadian_activity,
    plot_sentiment_evolution,
)

plot_sentiment_evolution(tweets, labels)
plt.show()

plot_circadian_activity(tweets, labels)
plt.show()

plot_activity_heatmap(tweets)
plt.show()

## 6. Atributos discriminantes

Comparação da distribuição de atributos selecionados entre as classes. Separação
visual sugere sinal aproveitável, mas não substitui a análise de importância.

In [ ]:
from visualization.distributions import plot_feature_distribution

for column in [
    "emo_negativo_ratio",
    "temp_night_activity_ratio",
    "ling_pronoun_first_singular",
]:
    if column in features.columns:
        plot_feature_distribution(features, column)
        plt.show()

## 7. Projeção semântica

**Leitura correta:** separação visual em 2D não implica separabilidade no espaço
original, nem o contrário. A figura é exploratória, não evidência de desempenho.

In [ ]:
from visualization.embeddings import plot_embedding_projection

figure = plot_embedding_projection(features, method="umap")
if figure is not None:
    plt.show()

## 8. Qualidade dos rótulos

A concordância entre a supervisão fraca e a revisão manual delimita o teto de
desempenho: nenhum modelo supera consistentemente a qualidade do rótulo com que
foi treinado.

In [ ]:
from utils.files import read_json

quality_path = PATHS.reports.metrics / "labeling_quality.json"
if quality_path.is_file():
    quality = read_json(quality_path)
    for key, value in quality.items():
        print(f"{key}: {value}")
else:
    print("Execute `make label` para gerar o relatório de qualidade dos rótulos.")